# Title: Engagement-Driven Content Generation Using Large Language Models

#### Sahil Arora (SID:501098147 ), Email: s11arora@torontomu.ca

#### Aman kaushik (SID: 501214203), Email: aman.kaushik@torontomu.ca

#### Evgenia Silajev (SID: 501017502), Email: esilajev@torontomu.ca

In our network, users have opinions distributed across a range. Extreme messages—very high or very low sentiment—only align with a small subset of users, so they activate fewer nodes. Moderate messages, on the other hand, are closer to the average opinion, so they fall within the acceptance threshold for more users in the bounded confidence model. That’s why they propagate more widely and achieve higher engagement.

We did use a lightweight LLM like GPT-2 for generating candidate posts, but we did not perform full LLM training or PPO fine-tuning. The original paper uses large models and reinforcement learning, which require GPU resources. Running that setup on a CPU-only environment would be extremely slow and impractical. So instead, we used a simplified approach where we map generated text to a numerical signal using sentiment and optimize that signal. This allowed us to capture the core idea while keeping the system computationally efficient, while also extending evaluation using multiple engagement metrics beyond sentiment.

Built a synthetic social network where each user has an opinion → to simulate real-world user diversity
Generated candidate messages using GPT-2 → to mimic realistic content creation
Converted text into a numerical signal using sentiment analysis → to make content measurable and usable in the model
Applied a bounded confidence diffusion model → to simulate how messages spread based on opinion similarity
Measured engagement using multiple metrics such as activation count, average opinion change, and reach ratio → to better capture both spread and influence
Performed best message search and PPO-style optimization → to find message values that maximize engagement
Used a simplified approach instead of full LLM training → because the original method requires GPU, so we used a CPU-friendly design while preserving the core idea

# Introduction

### Problem Description

The problem addressed in this project is how to generate social media content that maximizes user engagement within a network. Different users hold different opinions, and not all content spreads equally well. The key challenge is to identify which type of generated content can influence the largest number of users when propagated through a social network.

### Context of the Problem

This problem is important because social media platforms, marketing campaigns, and public communication strategies rely heavily on user engagement. Higher engagement leads to increased visibility, influence, and impact. Understanding how content spreads through a network—and how it interacts with user opinions—can significantly improve communication effectiveness in real-world scenarios.

### Limitation About Other Approaches

Traditional content generation methods focus mainly on text quality, relevance, or sentiment. However, they do not consider the structure of social networks or how information propagates between users. These approaches ignore the fact that different users respond differently depending on their existing opinions and their position within the network, leading to suboptimal engagement outcomes.

### Solution

In this project, we implement a simplified engagement-driven content generation framework that integrates language generation, sentiment analysis, network diffusion, and optimization. A synthetic social network is constructed where each user is assigned an opinion value. Multiple candidate posts are generated using a lightweight language model, and each post is converted into a numerical signal using sentiment scoring.

The generated content is then injected into the network and propagated using a bounded-confidence diffusion model, where users are influenced only if the message aligns closely with their opinions. Engagement is measured by the number of activated users, along with additional metrics such as average opinion change and reach ratio. We compare multiple generated posts, perform a search over possible message values, and apply a lightweight PPO-style optimization approach to iteratively improve the message for maximum engagement.

This approach moves beyond traditional text-based evaluation by incorporating network-aware dynamics, enabling the selection and optimization of content based on its ability to spread effectively through a social system.

# Background

Explain the related work using the following table

| Reference |Explanation |  Dataset/Input |Weakness
| --- | --- | --- | --- |
| Coppolillo et al. [1] (Engagement-Driven Content Generation with LLMs) |This method generates content using an LLM and evaluates it using a simulated social network. Engagement and fluency are used as reward signals to optimize content generation for maximum spread and influence.| Synthetic and real-world social networks (e.g., Brexit, Italian Referendum), prompts, node opinions, injection nodes | Computationally expensive and requires complex simulation and fine-tuning; future work can explore more realistic engagement models and scalable training methods

# Methodology
In this project, we implement a CPU-friendly simplified version of the engagement-driven content generation framework proposed by Coppolillo et al. Instead of reproducing the full large-scale reinforcement learning pipeline from the paper, we design a lightweight experimental system that preserves the core idea: content should be generated and selected based not only on textual quality, but also on its ability to spread through a social network and influence users.

Our method uses a synthetic social network dataset. First, a directed graph is created to represent a social network, where each node corresponds to a user and each edge represents a possible influence path. Each user is assigned an opinion value in the interval [0,1], where lower values represent more negative or resistant opinions and higher values represent more positive or receptive opinions.

Next, candidate posts are created using a small language model (gpt2) from a set of prompts. Each generated post is converted into a numeric signal using sentiment analysis. This sentiment score acts as the message value injected into the network. The message is then propagated through the graph using a simplified bounded-confidence diffusion rule: a user engages with the message if the difference between the user’s opinion and the message sentiment is below a fixed threshold. If engaged, the user’s opinion is partially updated toward the message, and the diffusion continues to neighboring nodes.

To evaluate engagement, we measure the total number of activated users after diffusion, along with additional metrics such as average opinion change and reach ratio. We then compare multiple generated posts, search across candidate message values, and apply a lightweight PPO-style iterative optimization that gradually updates the message toward values that produce higher engagement. This does not implement full policy-gradient PPO training; instead, it provides a computationally feasible approximation of reinforcement learning suitable for CPU-only execution on a Mac.

The complete pipeline is:

Prompt → Generate candidate posts → Score sentiment → Inject message into graph → Simulate diffusion → Measure activated users → Select or optimize the best message





# Implementation

In this project, we implement a simplified engagement-driven content generation system that runs efficiently on a Mac without GPU support. The system includes four main components: social network generation, message generation, diffusion, and optimization.

A synthetic directed graph is created where each node represents a user with an opinion value between 0 and 1. Candidate posts are generated using a lightweight language model (gpt2) and converted into numerical values using sentiment analysis.

These message values are propagated through the network using a bounded-confidence diffusion model, where users are influenced only if the message aligns with their opinions. Engagement is measured by the number of activated users, along with additional metrics such as average opinion change and reach ratio.

Finally, the system compares multiple messages, performs a search for the best message, and applies a lightweight PPO-style optimization to improve engagement. The graph is also visualized to illustrate how influence spreads across the network.

# 1: Import Libraries


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from transformers import pipeline
import random
import pandas as pd

## 2. Create Synthetic Social Network

In [ ]:
def create_graph(n=20, p=0.2, seed=42):
    np.random.seed(seed)
    random.seed(seed)

    G = nx.erdos_renyi_graph(n, p, directed=True, seed=seed)
    opinions = np.random.rand(n)

    for i in range(n):
        G.nodes[i]["opinion"] = float(opinions[i])

    return G, opinions

In [ ]:
# Import required libraries
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from transformers import pipeline
import random
import pandas as pd

## 3: Diffusion Model

In [ ]:
def propagate(G, opinions, message, start_node=0, threshold=0.3, alpha=0.5):
    updated_opinions = opinions.copy()
    visited = set([start_node])
    queue = [start_node]
    activated_nodes = set([start_node])

    while queue:
        node = queue.pop(0)

        for neighbor in G.neighbors(node):
            if neighbor not in visited:
                diff = abs(updated_opinions[neighbor] - message)

                if diff < threshold:
                    updated_opinions[neighbor] += alpha * (message - updated_opinions[neighbor])
                    activated_nodes.add(neighbor)
                    queue.append(neighbor)

                visited.add(neighbor)

    return len(activated_nodes), updated_opinions, activated_nodes

## 4. Additional Engagement Metrics (NEW SECTION) 

In [ ]:
def average_opinion_change(original_opinions, updated_opinions):
    return float(np.mean(np.abs(updated_opinions - original_opinions)))

def total_opinion_shift(original_opinions, updated_opinions):
    return float(np.sum(np.abs(updated_opinions - original_opinions)))

def reach_ratio(activated_nodes, total_nodes):
    return float(len(activated_nodes) / total_nodes)

def opinion_variance(opinions):
    return float(np.var(opinions))

## 5: Visualization Function

In [ ]:
def plot_graph(G, opinions, activated_nodes=None, title="Graph"):
    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=42)

    node_colors = opinions
    node_sizes = []

    for node in G.nodes():
        if activated_nodes is not None and node in activated_nodes:
            node_sizes.append(700)
        else:
            node_sizes.append(350)

    nx.draw(
        G,
        pos,
        with_labels=True,
        node_color=node_colors,
        node_size=node_sizes,
        cmap=plt.cm.coolwarm
    )

    plt.title(title)
    plt.show()

## 6. Message Generation & Sentiment

In [ ]:
generator = pipeline("text-generation", model="gpt2")
sentiment_model = pipeline("sentiment-analysis")

def generate_post(prompt="Climate action is"):
    result = generator(
        prompt,
        max_length=400,
        num_return_sequences=1,
        truncation=True
    )
    return result[0]["generated_text"]

def get_sentiment_score(text):
    result = sentiment_model(text)[0]
    label = result["label"]
    score = result["score"]

    if label.upper() == "POSITIVE":
        return float(score)
    else:
        return float(1 - score)

## 7. Multiple Message Comparison (UPDATED) ⭐

In [ ]:
def test_multiple_messages(G, opinions, prompts):
    records = []

    for i, prompt in enumerate(prompts, start=1):
        post = generate_post(prompt)
        sentiment_value = get_sentiment_score(post)

        activated_count, updated_opinions, activated_nodes = propagate(
            G, opinions, sentiment_value, start_node=0
        )

        avg_change = average_opinion_change(opinions, updated_opinions)
        total_shift = total_opinion_shift(opinions, updated_opinions)
        reach = reach_ratio(activated_nodes, len(G.nodes()))
        variance_after = opinion_variance(updated_opinions)

        records.append({
            "Post Number": i,
            "Prompt": prompt,
            "Post Text": post,
            "Sentiment Score": round(sentiment_value, 3),
            "Activation Count": activated_count,
            "Avg Opinion Change": round(avg_change, 3),
            "Total Opinion Shift": round(total_shift, 3),
            "Reach Ratio": round(reach, 3),
            "Opinion Variance After": round(variance_after, 3),
            "Activated Nodes": activated_nodes,
            "Updated Opinions": updated_opinions
        })

    return pd.DataFrame(records)

## 8. Best Message Search

In [ ]:
def search_best_message(G, opinions, resolution=21):
    best_score = -1
    best_message = None
    best_nodes = None
    best_updated = None

    for val in np.linspace(0, 1, resolution):
        activated_count, updated_opinions, activated_nodes = propagate(
            G, opinions, message=val, start_node=0
        )

        if activated_count > best_score:
            best_score = activated_count
            best_message = val
            best_nodes = activated_nodes
            best_updated = updated_opinions

    return best_message, best_score, best_nodes, best_updated

## 9. Lightweight PPO-style Optimization

In [ ]:
def lightweight_ppo(G, opinions, iterations=8, candidates_per_iter=6, step_size=0.12):
    current_message = random.random()
    history = []

    for it in range(iterations):
        candidates = [current_message]

        for _ in range(candidates_per_iter - 1):
            proposal = current_message + np.random.uniform(-step_size, step_size)
            proposal = min(max(proposal, 0.0), 1.0)
            candidates.append(proposal)

        candidate_scores = []
        for c in candidates:
            activated_count, _, _ = propagate(G, opinions, c, start_node=0)
            candidate_scores.append((c, activated_count))

        best_candidate, best_score = max(candidate_scores, key=lambda x: x[1])
        current_message = best_candidate

        history.append({
            "Iteration": it + 1,
            "Best Message": round(current_message, 3),
            "Best Engagement": best_score
        })

    return current_message, pd.DataFrame(history)

## 10. Main Execution (UPDATED) ⭐

In [ ]:
G, opinions = create_graph()

plot_graph(G, opinions, title="Initial Network")

prompts = [
    "Climate action is",
    "Clean energy future is",
    "Our community should",
    "Public policy must"
]

results_df = test_multiple_messages(G, opinions, prompts)

print(results_df[[
    "Post Number",
    "Sentiment Score",
    "Activation Count",
    "Avg Opinion Change",
    "Reach Ratio"
]])

# Best message
best_post_row = results_df.sort_values("Activation Count", ascending=False).iloc[0]

plot_graph(
    G,
    best_post_row["Updated Opinions"],
    activated_nodes=best_post_row["Activated Nodes"],
    title="Best Generated Message"
)

# Plot
plt.bar(results_df["Post Number"].astype(str), results_df["Activation Count"])
plt.title("Activation Count")
plt.show()

plt.bar(results_df["Post Number"].astype(str), results_df["Avg Opinion Change"])
plt.title("Average Opinion Change")
plt.show()

# Conclusion and Future Direction

In this project, we implemented a simplified engagement-driven content generation system using a synthetic social network and diffusion model. The results and graph visualization show that content effectiveness depends on how well the message aligns with user opinions and network structure. Messages closer to the overall opinion distribution activate more users and achieve higher engagement, while additional metrics such as opinion change and reach provide further insight into influence.

The main strength of this approach is that it captures network-aware content optimization in a computationally efficient way. However, the model has limitations, including the use of synthetic data, simplified sentiment scoring, and a lightweight approximation of reinforcement learning.

In future work, this system can be improved by using real-world datasets, more advanced language models, and full reinforcement learning methods. Additional enhancements could include modeling dynamic user behavior and more realistic engagement metrics.

# References:

[1]: Erica Coppolillo, Federico Cinus, Marco Minici, Francesco Bonchi, and Giuseppe Manco, Engagement-Driven Content Generation with Large Language Models, Proceedings of the 31st ACM SIGKDD Conference on Knowledge Discovery and Data Mining (KDD), 2025.

